# Module 01 — Mathematical & Programming Foundations## 01-10: Computational Thinking & Complexity**Objective:** Develop the ability to analyze algorithm efficiency —Big-O notation, time/space complexity, vectorization gains, and memoryhierarchy effects — so you can predict bottlenecks and write scalableML code.**Prerequisites:** 01-01 (Python, NumPy & Tensor Speed), 01-02 (Advanced NumPy & PyTorch Operations), 01-09 (Calculus & Optimization Foundations)

---## Part 0 — Setup & PrerequisitesUnderstanding computational complexity is essential for ML practitioners.Choosing between $O(n^2)$ and $O(n \log n)$ attention mechanisms, decidingwhen to vectorize versus loop, and understanding GPU memory layout are alldecisions that depend on complexity analysis.This notebook covers:- **Big-O notation** — classifying growth rates of algorithms- **Empirical complexity measurement** — timing code and fitting growth curves- **Common ML algorithm complexities** — training vs inference costs- **Vectorization analysis** — why NumPy/PyTorch beat Python loops- **Memory hierarchy** — cache, RAM, GPU memory effects on performance- **Space complexity** — memory footprint of models and data structures**Prerequisites:** 01-01, 01-02, 01-09

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import time
from collections import defaultdict

print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'NumPy: {np.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────────────
import random

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
FIGSIZE = (10, 6)
COLORS = {
    'blue': '#1E88E5',
    'red': '#E53935',
    'green': '#43A047',
    'orange': '#FF9800',
    'purple': '#9C27B0',
    'teal': '#00897B',
    'gray': '#757575',
}
COLOR_LIST = list(COLORS.values())

### Timing UtilityWe define a helper to measure execution time reliably acrossmultiple runs.

In [ ]:
def time_function(
    func: callable,
    *args,
    n_runs: int = 5,
    **kwargs,
) -> tuple[float, float]:
    """Time a function call over multiple runs.

    Args:
        func: Function to time.
        *args: Positional arguments passed to func.
        n_runs: Number of repetitions.
        **kwargs: Keyword arguments passed to func.

    Returns:
        Tuple of (mean_time_seconds, std_time_seconds).
    """
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
    return np.mean(times), np.std(times)

---## Part 1 — Computational Complexity from ScratchWe build up from Big-O definitions to empirical complexity measurementand analysis of common ML algorithms.

### 1.1 Big-O NotationBig-O describes the **upper bound** on growth rate as input size $n \to \infty$:$$f(n) = O(g(n)) \iff \exists\, c > 0,\, n_0 \text{ such that } f(n) \leq c \cdot g(n) \; \forall\, n \geq n_0$$Common complexity classes (fastest to slowest):| Complexity | Name | Example ||-----------|------|--------|| $O(1)$ | Constant | Array index lookup || $O(\log n)$ | Logarithmic | Binary search || $O(n)$ | Linear | Linear scan, single-pass statistics || $O(n \log n)$ | Linearithmic | Efficient sorting (mergesort) || $O(n^2)$ | Quadratic | Pairwise distance matrix, naive attention || $O(n^3)$ | Cubic | Matrix multiplication (naive), matrix inversion || $O(2^n)$ | Exponential | Brute-force subset enumeration |

In [ ]:
def visualize_growth_rates() -> None:
    """Plot common Big-O growth rates side by side."""
    n = np.arange(1, 101)

    complexities = {
        '$O(1)$': np.ones_like(n, dtype=float),
        '$O(\log n)$': np.log2(n),
        '$O(n)$': n.astype(float),
        '$O(n \log n)$': n * np.log2(n),
        '$O(n^2)$': n.astype(float) ** 2,
        '$O(n^3)$': n.astype(float) ** 3,
    }

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Linear scale (shows how quickly n² and n³ dominate)
    for (label, values), color in zip(complexities.items(), COLOR_LIST):
        axes[0].plot(n, values, color=color, linewidth=2, label=label)
    axes[0].set_xlabel('Input Size n', fontsize=12)
    axes[0].set_ylabel('Operations', fontsize=12)
    axes[0].set_title('Growth Rates (Linear Scale)')
    axes[0].legend(fontsize=10)
    axes[0].set_ylim(0, 5000)
    axes[0].grid(True, alpha=0.2)

    # Log-log scale (shows the slopes = exponents)
    for (label, values), color in zip(complexities.items(), COLOR_LIST):
        axes[1].loglog(n, values, color=color, linewidth=2, label=label)
    axes[1].set_xlabel('Input Size n', fontsize=12)
    axes[1].set_ylabel('Operations (log)', fontsize=12)
    axes[1].set_title('Growth Rates (Log-Log Scale)\nSlope = exponent')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.2)

    plt.suptitle('Big-O Complexity Classes', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Practical impact table
    print('\nTime for different n (assuming 1 ns per operation):')
    practical_n = [100, 1000, 10000, 100000]
    rows = []
    for size in practical_n:
        rows.append({
            'n': f'{size:,}',
            'O(n)': f'{size * 1e-9:.2e}s',
            'O(n log n)': f'{size * np.log2(size) * 1e-9:.2e}s',
            'O(n²)': f'{size**2 * 1e-9:.2e}s',
            'O(n³)': f'{size**3 * 1e-9:.2e}s',
        })
    print(pd.DataFrame(rows).to_string(index=False))


visualize_growth_rates()

### 1.2 Empirical Complexity MeasurementInstead of analyzing code theoretically, we can **measure** the runtimeat different input sizes and fit a growth curve. On a log-log plot,$O(n^k)$ appears as a line with slope $k$.

In [ ]:
def measure_complexity(
    func: callable,
    sizes: list[int],
    setup: callable | None = None,
    n_runs: int = 3,
) -> tuple[list[int], list[float]]:
    """Measure empirical time complexity of a function.

    Args:
        func: Function that takes (data, n) as arguments.
        sizes: List of input sizes to test.
        setup: Optional function to create input data: setup(n) -> data.
        n_runs: Number of repetitions per size.

    Returns:
        Tuple of (sizes, mean_times).
    """
    times_list = []
    for n in sizes:
        data = setup(n) if setup else None
        mean_t, _ = time_function(func, data, n, n_runs=n_runs)
        times_list.append(mean_t)
    return sizes, times_list


def fit_complexity(
    sizes: list[int],
    times: list[float],
) -> tuple[float, float]:
    """Fit O(n^k) by linear regression on log-log data.

    Args:
        sizes: Input sizes.
        times: Measured times.

    Returns:
        Tuple of (exponent_k, log_constant).
    """
    log_n = np.log(np.array(sizes, dtype=float))
    log_t = np.log(np.array(times, dtype=float))
    # Linear regression: log_t = k * log_n + c
    coeffs = np.polyfit(log_n, log_t, 1)
    return coeffs[0], coeffs[1]


# Measure known algorithms
def algo_linear(data: np.ndarray | None, n: int) -> None:
    """O(n) — sum of array.

    Args:
        data: Input array.
        n: Size (unused, data already sized).
    """
    total = 0.0
    for val in data:
        total += val
    _ = total


def algo_quadratic(data: np.ndarray | None, n: int) -> None:
    """O(n²) — pairwise comparison.

    Args:
        data: Input array.
        n: Size.
    """
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if data[i] > data[j]:
                count += 1
    _ = count


def algo_nlogn(data: np.ndarray | None, n: int) -> None:
    """O(n log n) — sorting.

    Args:
        data: Input array.
        n: Size (unused).
    """
    _ = sorted(data)


rng_setup = np.random.RandomState(SEED)
setup_fn = lambda n: rng_setup.randn(n)

# Keep sizes small enough for O(n²) in Python
sizes_small = [100, 200, 500, 1000, 2000, 5000]
sizes_medium = [100, 500, 1000, 5000, 10000, 50000]

print('Measuring O(n) — sum...')
_, times_linear = measure_complexity(algo_linear, sizes_medium, setup_fn)
k_linear, _ = fit_complexity(sizes_medium, times_linear)

print('Measuring O(n log n) — sort...')
_, times_nlogn = measure_complexity(algo_nlogn, sizes_medium, setup_fn)
k_nlogn, _ = fit_complexity(sizes_medium, times_nlogn)

print('Measuring O(n²) — pairwise...')
_, times_quadratic = measure_complexity(algo_quadratic, sizes_small, setup_fn)
k_quadratic, _ = fit_complexity(sizes_small, times_quadratic)

print(f'\nFitted exponents (O(n^k)):')
print(f'  Linear sum:      k = {k_linear:.2f}  (expected ~1.0)')
print(f'  Sort:            k = {k_nlogn:.2f}  (expected ~1.0-1.1)')
print(f'  Pairwise:        k = {k_quadratic:.2f}  (expected ~2.0)')

In [ ]:
def plot_empirical_complexity() -> None:
    """Plot measured complexity on log-log axes."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Log-log plot
    axes[0].loglog(sizes_medium, times_linear, 'o-', color=COLORS['blue'],
                    linewidth=2, markersize=6, label=f'Sum: k={k_linear:.2f}')
    axes[0].loglog(sizes_medium, times_nlogn, 's-', color=COLORS['green'],
                    linewidth=2, markersize=6, label=f'Sort: k={k_nlogn:.2f}')
    axes[0].loglog(sizes_small, times_quadratic, '^-', color=COLORS['red'],
                    linewidth=2, markersize=6, label=f'Pairwise: k={k_quadratic:.2f}')

    # Reference slopes
    ref_n = np.array([100, 50000], dtype=float)
    axes[0].loglog(ref_n, ref_n * 1e-8, '--', color=COLORS['gray'], alpha=0.4, label='O(n)')
    axes[0].loglog(ref_n, ref_n**2 * 1e-10, '--', color=COLORS['orange'], alpha=0.4, label='O(n²)')

    axes[0].set_xlabel('Input Size n', fontsize=12)
    axes[0].set_ylabel('Time (seconds)', fontsize=12)
    axes[0].set_title('Empirical Complexity (Log-Log)')
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.2)

    # Ratio test: t(2n) / t(n) reveals complexity
    # O(n) → ratio ≈ 2, O(n²) → ratio ≈ 4, O(n³) → ratio ≈ 8
    ratios_linear = [times_linear[i+1] / times_linear[i]
                      for i in range(len(times_linear) - 1)]
    size_ratios_lin = [sizes_medium[i+1] / sizes_medium[i]
                       for i in range(len(sizes_medium) - 1)]

    ratios_quad = [times_quadratic[i+1] / times_quadratic[i]
                    for i in range(len(times_quadratic) - 1)]
    size_ratios_quad = [sizes_small[i+1] / sizes_small[i]
                        for i in range(len(sizes_small) - 1)]

    # Plot time ratios vs size ratios
    axes[1].scatter(size_ratios_lin, ratios_linear, s=80, color=COLORS['blue'],
                     label='Sum (expect ratio ≈ size_ratio)', zorder=5)
    axes[1].scatter(size_ratios_quad, ratios_quad, s=80, color=COLORS['red'],
                     label='Pairwise (expect ratio ≈ size_ratio²)', zorder=5)

    # Reference lines
    sr = np.linspace(1, 10, 50)
    axes[1].plot(sr, sr, '--', color=COLORS['blue'], alpha=0.4, label='O(n): ratio = r')
    axes[1].plot(sr, sr**2, '--', color=COLORS['red'], alpha=0.4, label='O(n²): ratio = r²')

    axes[1].set_xlabel('Size Ratio (n₂/n₁)', fontsize=12)
    axes[1].set_ylabel('Time Ratio (t₂/t₁)', fontsize=12)
    axes[1].set_title('Ratio Test: Time Ratio vs Size Ratio')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.2)

    plt.suptitle('Empirical Complexity Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


plot_empirical_complexity()

### 1.3 Complexity of Common ML AlgorithmsUnderstanding the computational cost of ML algorithms helps you choosethe right tool for the job. Here $n$ = samples, $d$ = features,$K$ = classes/clusters, $T$ = trees.

In [ ]:
def ml_algorithm_complexity_table() -> None:
    """Display complexity of common ML algorithms."""
    algorithms = [
        {'Algorithm': 'Linear Regression (closed-form)', 'Training': 'O(nd² + d³)',
         'Prediction': 'O(d)', 'Space': 'O(d²)'},
        {'Algorithm': 'Linear Regression (GD, k iters)', 'Training': 'O(knd)',
         'Prediction': 'O(d)', 'Space': 'O(d)'},
        {'Algorithm': 'k-NN', 'Training': 'O(1) (lazy)',
         'Prediction': 'O(nd)', 'Space': 'O(nd)'},
        {'Algorithm': 'Decision Tree', 'Training': 'O(nd log n)',
         'Prediction': 'O(log n)', 'Space': 'O(nodes)'},
        {'Algorithm': 'Random Forest (T trees)', 'Training': 'O(Tnd log n)',
         'Prediction': 'O(T log n)', 'Space': 'O(T × nodes)'},
        {'Algorithm': 'SVM (kernel)', 'Training': 'O(n²d ~ n³)',
         'Prediction': 'O(n_sv × d)', 'Space': 'O(n²)'},
        {'Algorithm': 'K-Means (k clusters, t iters)', 'Training': 'O(tnKd)',
         'Prediction': 'O(Kd)', 'Space': 'O(Kd)'},
        {'Algorithm': 'PCA (truncated, r components)', 'Training': 'O(ndr)',
         'Prediction': 'O(dr)', 'Space': 'O(dr)'},
        {'Algorithm': 'Fully Connected NN (L layers)', 'Training': 'O(n × Σ hᵢhᵢ₊₁)',
         'Prediction': 'O(Σ hᵢhᵢ₊₁)', 'Space': 'O(Σ hᵢhᵢ₊₁)'},
        {'Algorithm': 'Transformer (seq len S)', 'Training': 'O(nS²d)',
         'Prediction': 'O(S²d)', 'Space': 'O(S² + Sd)'},
    ]

    df = pd.DataFrame(algorithms)
    print(df.to_string(index=False))

    print('\nKey observations:')
    print('  - k-NN has O(1) training but O(nd) prediction — lazy learning')
    print('  - Transformer attention is O(S²) — quadratic in sequence length')
    print('  - Closed-form linear regression has O(d³) — cubic in features')
    print('  - GD-based training scales linearly with iterations and samples')


ml_algorithm_complexity_table()

### 1.4 Vectorization AnalysisNumPy and PyTorch achieve massive speedups over Python loops by:1. **BLAS/LAPACK** — optimized low-level linear algebra (C/Fortran)2. **SIMD instructions** — process multiple values in a single CPU cycle3. **Avoiding Python overhead** — no interpreter per-element4. **Cache-friendly memory access** — contiguous arraysLet's measure the speedup for key ML operations.

In [ ]:
def vectorization_benchmarks() -> None:
    """Compare Python loops vs NumPy vs PyTorch for common operations."""
    rng = np.random.RandomState(SEED)
    results = []

    # ── Dot product ──
    n = 100000
    a = rng.randn(n)
    b = rng.randn(n)
    a_t = torch.tensor(a)
    b_t = torch.tensor(b)

    def dot_python(a: np.ndarray, b: np.ndarray, n: int) -> None:
        """Python loop dot product."""
        total = 0.0
        for i in range(n):
            total += a[i] * b[i]
        _ = total

    t_py, _ = time_function(dot_python, a, b, n)
    t_np, _ = time_function(lambda a, b, n: np.dot(a, b), a, b, n)
    t_pt, _ = time_function(lambda a, b, n: torch.dot(a_t, b_t), a_t, b_t, n)

    results.append({'Operation': f'Dot product (n={n:,})',
                    'Python (ms)': f'{t_py*1000:.2f}',
                    'NumPy (ms)': f'{t_np*1000:.4f}',
                    'PyTorch (ms)': f'{t_pt*1000:.4f}',
                    'Speedup (NumPy)': f'{t_py/t_np:.0f}x'})

    # ── Matrix multiplication ──
    n_mat = 500
    A = rng.randn(n_mat, n_mat)
    B = rng.randn(n_mat, n_mat)
    A_t = torch.tensor(A)
    B_t = torch.tensor(B)

    def matmul_python(A: np.ndarray, B: np.ndarray, n: int) -> None:
        """Python triple-loop matrix multiplication."""
        C = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                for k in range(n):
                    C[i, j] += A[i, k] * B[k, j]
        _ = C

    # Only run Python matmul for small size
    n_small_mat = 100
    A_small = A[:n_small_mat, :n_small_mat]
    B_small = B[:n_small_mat, :n_small_mat]
    t_py_mat, _ = time_function(matmul_python, A_small, B_small, n_small_mat, n_runs=1)
    t_np_mat, _ = time_function(lambda A, B, n: A @ B, A, B, n_mat)
    t_pt_mat, _ = time_function(lambda A, B, n: A_t @ B_t, A_t, B_t, n_mat)

    # Extrapolate Python time for n=500
    scale_factor = (n_mat / n_small_mat) ** 3
    t_py_mat_est = t_py_mat * scale_factor

    results.append({'Operation': f'Matrix multiply ({n_mat}×{n_mat})',
                    'Python (ms)': f'{t_py_mat_est*1000:.0f} (est)',
                    'NumPy (ms)': f'{t_np_mat*1000:.2f}',
                    'PyTorch (ms)': f'{t_pt_mat*1000:.2f}',
                    'Speedup (NumPy)': f'{t_py_mat_est/t_np_mat:.0f}x (est)'})

    # ── Element-wise operations ──
    n_elem = 1000000
    arr = rng.randn(n_elem)
    arr_t = torch.tensor(arr)

    def elementwise_python(arr: np.ndarray, n: int) -> None:
        """Python loop element-wise square."""
        result = [x ** 2 for x in arr[:n]]
        _ = result

    t_py_elem, _ = time_function(elementwise_python, arr[:100000], 100000)
    t_np_elem, _ = time_function(lambda a, n: a ** 2, arr, n_elem)
    t_pt_elem, _ = time_function(lambda a, n: a ** 2, arr_t, n_elem)

    # Scale Python time
    t_py_elem_scaled = t_py_elem * (n_elem / 100000)

    results.append({'Operation': f'Element-wise x² (n={n_elem:,})',
                    'Python (ms)': f'{t_py_elem_scaled*1000:.1f} (est)',
                    'NumPy (ms)': f'{t_np_elem*1000:.3f}',
                    'PyTorch (ms)': f'{t_pt_elem*1000:.3f}',
                    'Speedup (NumPy)': f'{t_py_elem_scaled/t_np_elem:.0f}x (est)'})

    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False))


vectorization_benchmarks()

### 1.5 Broadcasting: Hidden Memory CostsNumPy broadcasting is powerful but can create large intermediate arraysthat consume memory. Understanding when broadcasting allocates memoryvs when it's free is critical for ML at scale.

In [ ]:
def broadcasting_analysis() -> None:
    """Analyze memory and time costs of broadcasting."""
    rng = np.random.RandomState(SEED)

    # Pairwise distance: ||x_i - x_j||²
    sizes = [100, 500, 1000, 2000, 5000]
    d = 10

    times_broadcast = []
    times_loop = []
    memory_broadcast = []

    for n in sizes:
        X = rng.randn(n, d)

        # Broadcasting approach: (n, 1, d) - (1, n, d) → (n, n, d)
        def pairwise_broadcast(X: np.ndarray, n: int) -> np.ndarray:
            """Pairwise distances via broadcasting."""
            diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
            return np.sum(diff ** 2, axis=2)

        # Loop approach
        def pairwise_loop(X: np.ndarray, n: int) -> np.ndarray:
            """Pairwise distances via loop."""
            dists = np.zeros((n, n))
            for i in range(n):
                dists[i] = np.sum((X[i] - X) ** 2, axis=1)
            return dists

        if n <= 2000:
            t_bc, _ = time_function(pairwise_broadcast, X, n, n_runs=3)
            times_broadcast.append(t_bc)
            # Peak memory: n × n × d float64 = n²d × 8 bytes
            mem_mb = n * n * d * 8 / (1024 ** 2)
            memory_broadcast.append(mem_mb)
        else:
            times_broadcast.append(np.nan)
            memory_broadcast.append(n * n * d * 8 / (1024 ** 2))

        t_lp, _ = time_function(pairwise_loop, X, n, n_runs=3)
        times_loop.append(t_lp)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].loglog(sizes[:4], times_broadcast[:4], 'o-', color=COLORS['blue'],
                    linewidth=2, label='Broadcasting')
    axes[0].loglog(sizes, times_loop, 's-', color=COLORS['red'],
                    linewidth=2, label='Loop')
    axes[0].set_xlabel('n (samples)')
    axes[0].set_ylabel('Time (seconds)')
    axes[0].set_title('Pairwise Distance: Time')
    axes[0].legend()
    axes[0].grid(True, alpha=0.2)

    axes[1].semilogy(sizes, memory_broadcast, 'o-', color=COLORS['orange'],
                      linewidth=2, label='Peak intermediate memory')
    axes[1].axhline(1000, color=COLORS['red'], linestyle='--', alpha=0.5,
                     label='1 GB')
    axes[1].axhline(8000, color=COLORS['red'], linestyle=':', alpha=0.5,
                     label='8 GB')
    axes[1].set_xlabel('n (samples)')
    axes[1].set_ylabel('Memory (MB)')
    axes[1].set_title('Broadcasting: Hidden Memory Cost\n'
                       'Intermediate (n×n×d) can blow up!')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.2)

    plt.suptitle('Broadcasting Trade-off: Speed vs Memory',
                  fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Peak memory for pairwise broadcast (n×n×d×8 bytes, d=10):')
    for n, mem in zip(sizes, memory_broadcast):
        print(f'  n={n:>5,}: {mem:>10.1f} MB')


broadcasting_analysis()

### 1.6 Memory Layout: Row-Major vs Column-MajorHow data is stored in memory affects cache performance. NumPy defaultsto **row-major (C order)** while Fortran uses **column-major (F order)**.Accessing data in memory order is much faster due to CPU cache lines.

In [ ]:
def memory_layout_benchmark() -> None:
    """Benchmark row vs column access patterns."""
    rng = np.random.RandomState(SEED)
    n = 5000
    A_c = np.ascontiguousarray(rng.randn(n, n))  # Row-major (C)
    A_f = np.asfortranarray(A_c)                   # Column-major (F)

    # Row sum (fast for C order, slow for F order)
    t_c_row, _ = time_function(lambda A, n: A.sum(axis=1), A_c, n)
    t_f_row, _ = time_function(lambda A, n: A.sum(axis=1), A_f, n)

    # Column sum (fast for F order, slow for C order)
    t_c_col, _ = time_function(lambda A, n: A.sum(axis=0), A_c, n)
    t_f_col, _ = time_function(lambda A, n: A.sum(axis=0), A_f, n)

    fig, ax = plt.subplots(figsize=(10, 5))
    x_pos = np.arange(2)
    width = 0.25

    ax.bar(x_pos - width, [t_c_row * 1000, t_c_col * 1000], width,
            color=COLORS['blue'], alpha=0.7, label='C order (row-major)')
    ax.bar(x_pos, [t_f_row * 1000, t_f_col * 1000], width,
            color=COLORS['red'], alpha=0.7, label='F order (column-major)')

    ax.set_xticks(x_pos - width / 2)
    ax.set_xticklabels(['Row Sum (axis=1)', 'Column Sum (axis=0)'])
    ax.set_ylabel('Time (ms)')
    ax.set_title(f'Memory Layout Effect on Performance (n={n})')
    ax.legend()
    ax.grid(True, alpha=0.2, axis='y')
    plt.tight_layout()
    plt.show()

    print(f'Row sum — C order: {t_c_row*1000:.2f} ms, F order: {t_f_row*1000:.2f} ms')
    print(f'Col sum — C order: {t_c_col*1000:.2f} ms, F order: {t_f_col*1000:.2f} ms')
    print(f'Rule: Access data along the contiguous dimension for best cache performance')
    print(f'PyTorch uses row-major by default — process along last dimension when possible')


memory_layout_benchmark()

### 1.7 Space Complexity: Model Memory FootprintUnderstanding memory usage is critical for fitting models on GPUs.Key contributors:- **Model parameters:** weights and biases- **Activations:** stored for backpropagation (scales with batch size)- **Gradients:** same size as parameters- **Optimizer state:** 1x for SGD, 2x for Adam (moments)

In [ ]:
def model_memory_analysis() -> None:
    """Analyze memory footprint of neural network models."""
    def count_parameters(layer_sizes: list[int]) -> int:
        """Count parameters in a fully-connected network.

        Args:
            layer_sizes: List of layer widths including input/output.

        Returns:
            Total parameter count.
        """
        total = 0
        for i in range(len(layer_sizes) - 1):
            # Weights + biases
            total += layer_sizes[i] * layer_sizes[i + 1] + layer_sizes[i + 1]
        return total

    def memory_breakdown(
        n_params: int,
        batch_size: int,
        activation_size: int,
        dtype_bytes: int = 4,
        optimizer: str = 'adam',
    ) -> dict[str, float]:
        """Compute memory breakdown for training.

        Args:
            n_params: Number of model parameters.
            batch_size: Training batch size.
            activation_size: Total activation elements per sample.
            dtype_bytes: Bytes per element (4 for float32).
            optimizer: Optimizer type ('sgd' or 'adam').

        Returns:
            Dictionary of memory components in MB.
        """
        param_mb = n_params * dtype_bytes / (1024 ** 2)
        grad_mb = param_mb  # Same size as parameters
        act_mb = batch_size * activation_size * dtype_bytes / (1024 ** 2)

        opt_multiplier = 2 if optimizer == 'adam' else 0
        opt_mb = opt_multiplier * param_mb

        return {
            'Parameters': param_mb,
            'Gradients': grad_mb,
            'Activations': act_mb,
            'Optimizer State': opt_mb,
            'Total': param_mb + grad_mb + act_mb + opt_mb,
        }

    # Example architectures
    architectures = [
        ('Small MLP', [784, 256, 128, 10], 32, 256 + 128 + 10),
        ('Large MLP', [784, 1024, 512, 256, 10], 64, 1024 + 512 + 256 + 10),
        ('ResNet-18 (approx)', [3*224*224], 32, 11_700_000),  # 11.7M params
        ('GPT-2 Small (approx)', [768], 8, 117_000_000),       # 117M params
    ]

    rows = []
    for name, layers, batch_sz, act_sz in architectures:
        if len(layers) > 1:
            n_params = count_parameters(layers)
        else:
            # Use act_sz as param count for approximate models
            n_params = act_sz
            act_sz = n_params // 4  # Rough activation estimate

        breakdown = memory_breakdown(n_params, batch_sz, act_sz)
        rows.append({
            'Model': name,
            'Parameters': f'{n_params:,}',
            'Param (MB)': f"{breakdown['Parameters']:.1f}",
            'Grad (MB)': f"{breakdown['Gradients']:.1f}",
            'Act (MB)': f"{breakdown['Activations']:.1f}",
            'Opt (MB)': f"{breakdown['Optimizer State']:.1f}",
            'Total (MB)': f"{breakdown['Total']:.1f}",
        })

    mem_df = pd.DataFrame(rows)
    print(mem_df.to_string(index=False))

    # Visualize scaling
    param_counts = [100_000, 1_000_000, 10_000_000, 100_000_000, 1_000_000_000]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for batch_sz, color in [(8, COLORS['blue']), (32, COLORS['green']),
                             (128, COLORS['red'])]:
        totals = []
        for n_p in param_counts:
            breakdown = memory_breakdown(n_p, batch_sz, n_p // 4)
            totals.append(breakdown['Total'])
        axes[0].loglog(param_counts, totals, 'o-', color=color,
                        linewidth=2, label=f'Batch={batch_sz}')

    axes[0].axhline(4000, color=COLORS['gray'], linestyle='--', alpha=0.5,
                     label='4 GB (typical GPU)')
    axes[0].axhline(16000, color=COLORS['gray'], linestyle=':', alpha=0.5,
                     label='16 GB (V100/A100)')
    axes[0].set_xlabel('Number of Parameters')
    axes[0].set_ylabel('Total Memory (MB)')
    axes[0].set_title('Training Memory vs Model Size')
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.2)

    # Memory breakdown pie chart for a typical model
    breakdown_example = memory_breakdown(10_000_000, 32, 2_500_000)
    labels = ['Parameters', 'Gradients', 'Activations', 'Optimizer']
    values = [breakdown_example[k] for k in ['Parameters', 'Gradients',
              'Activations', 'Optimizer State']]
    axes[1].pie(values, labels=labels, autopct='%1.0f%%',
                 colors=[COLORS['blue'], COLORS['green'],
                         COLORS['orange'], COLORS['red']])
    axes[1].set_title(f'Memory Breakdown (10M param model)\n'
                       f'Total: {breakdown_example["Total"]:.0f} MB')

    plt.tight_layout()
    plt.show()


model_memory_analysis()

---## Part 2 — Putting It All Together: ComplexityAnalyzerWe assemble our analysis tools into a reusable class.

In [ ]:
class ComplexityAnalyzer:
    """Tool for empirically measuring and classifying algorithm complexity.

    Attributes:
        sizes: Input sizes tested.
        times: Measured execution times.
        fitted_exponent: Fitted power-law exponent.
    """

    def __init__(self) -> None:
        """Initialize analyzer."""
        self.sizes: list[int] = []
        self.times: list[float] = []
        self.fitted_exponent: float = 0.0
        self._log_constant: float = 0.0

    def measure(
        self,
        func: callable,
        sizes: list[int],
        setup: callable | None = None,
        n_runs: int = 3,
    ) -> None:
        """Measure execution time at different input sizes.

        Args:
            func: Function to measure (takes data, n).
            sizes: Input sizes.
            setup: Data creation function.
            n_runs: Repetitions per size.
        """
        self.sizes, self.times = measure_complexity(func, sizes, setup, n_runs)
        self.fitted_exponent, self._log_constant = fit_complexity(
            self.sizes, self.times
        )

    def classify(self) -> str:
        """Classify the measured complexity into standard Big-O classes.

        Returns:
            String description of the complexity class.
        """
        k = self.fitted_exponent
        if k < 0.3:
            return 'O(1) — Constant'
        elif k < 0.7:
            return 'O(√n) — Sub-linear'
        elif k < 1.3:
            return 'O(n) — Linear'
        elif k < 1.7:
            return 'O(n log n) — Linearithmic'
        elif k < 2.3:
            return 'O(n²) — Quadratic'
        elif k < 3.3:
            return 'O(n³) — Cubic'
        else:
            return f'O(n^{k:.1f}) — Polynomial'

    def report(self) -> pd.DataFrame:
        """Generate a summary report.

        Returns:
            DataFrame with sizes, times, and analysis.
        """
        df = pd.DataFrame({
            'Size': self.sizes,
            'Time (ms)': [t * 1000 for t in self.times],
        })
        return df

    def plot(self, title: str = 'Complexity Analysis') -> None:
        """Plot measured times with fitted line on log-log axes.

        Args:
            title: Plot title.
        """
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.loglog(self.sizes, self.times, 'o-', color=COLORS['blue'],
                   linewidth=2, markersize=8, label='Measured')

        # Fitted line
        n_fit = np.linspace(min(self.sizes), max(self.sizes), 100)
        t_fit = np.exp(self._log_constant) * n_fit ** self.fitted_exponent
        ax.loglog(n_fit, t_fit, '--', color=COLORS['red'],
                   linewidth=2, label=f'Fit: O(n^{self.fitted_exponent:.2f})')

        ax.set_xlabel('Input Size n', fontsize=12)
        ax.set_ylabel('Time (seconds)', fontsize=12)
        ax.set_title(f'{title}\nClassified as: {self.classify()}')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.2)
        plt.tight_layout()
        plt.show()


# Sanity check
analyzer = ComplexityAnalyzer()
analyzer.measure(
    algo_quadratic, [100, 200, 500, 1000, 2000],
    setup=lambda n: np.random.RandomState(SEED).randn(n),
)
print(f'Exponent: {analyzer.fitted_exponent:.2f}')
print(f'Classification: {analyzer.classify()}')
assert 1.5 < analyzer.fitted_exponent < 2.5, 'Expected quadratic!'
print('Sanity check PASSED')

---## Part 3 — Application: Analyzing Real ML OperationsLet's use our tools to analyze the complexity of real ML operationsthat matter in practice.

### 3.1 Matrix Operations Scaling

In [ ]:
def matrix_operations_complexity() -> None:
    """Measure complexity of key matrix operations."""
    rng = np.random.RandomState(SEED)

    operations = {
        'Matrix-Vector (Ax)': {
            'func': lambda data, n: data[0] @ data[1],
            'setup': lambda n: (rng.randn(n, n), rng.randn(n)),
            'expected': 'O(n²)',
        },
        'Matrix Multiply (AB)': {
            'func': lambda data, n: data[0] @ data[1],
            'setup': lambda n: (rng.randn(n, n), rng.randn(n, n)),
            'expected': 'O(n³)',
        },
        'SVD': {
            'func': lambda data, n: np.linalg.svd(data, full_matrices=False),
            'setup': lambda n: rng.randn(n, n),
            'expected': 'O(n³)',
        },
        'Eigendecomposition': {
            'func': lambda data, n: np.linalg.eigh(data),
            'setup': lambda n: (lambda A: A @ A.T)(rng.randn(n, n)),
            'expected': 'O(n³)',
        },
    }

    sizes_mat = [50, 100, 200, 400, 800]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()

    summary_rows = []
    for idx, (name, config) in enumerate(operations.items()):
        analyzer_op = ComplexityAnalyzer()
        analyzer_op.measure(config['func'], sizes_mat, config['setup'], n_runs=3)

        axes[idx].loglog(analyzer_op.sizes, analyzer_op.times, 'o-',
                          color=COLORS['blue'], linewidth=2, markersize=6)

        n_fit = np.linspace(min(sizes_mat), max(sizes_mat), 50)
        t_fit = np.exp(analyzer_op._log_constant) * n_fit ** analyzer_op.fitted_exponent
        axes[idx].loglog(n_fit, t_fit, '--', color=COLORS['red'],
                          linewidth=1.5, alpha=0.7)

        axes[idx].set_xlabel('Matrix Size n')
        axes[idx].set_ylabel('Time (s)')
        axes[idx].set_title(f'{name}\nMeasured: O(n^{analyzer_op.fitted_exponent:.2f}), '
                             f'Expected: {config["expected"]}')
        axes[idx].grid(True, alpha=0.2)

        summary_rows.append({
            'Operation': name,
            'Expected': config['expected'],
            'Measured k': f'{analyzer_op.fitted_exponent:.2f}',
            'Time at n=800': f'{analyzer_op.times[-1]*1000:.1f} ms',
        })

    plt.suptitle('Matrix Operation Complexity', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(pd.DataFrame(summary_rows).to_string(index=False))
    print('\nNote: NumPy uses optimized BLAS (Strassen-like), '
          'so measured exponent may be < 3.0')


matrix_operations_complexity()

### 3.2 Attention Complexity: O(n²) ProblemSelf-attention in Transformers has $O(S^2 d)$ complexity where $S$ issequence length. This quadratic scaling is the main bottleneck forlong-sequence models.

In [ ]:
def attention_complexity_demo() -> None:
    """Demonstrate quadratic complexity of self-attention."""
    rng = np.random.RandomState(SEED)
    d_model = 64

    def naive_attention(data: np.ndarray, n: int) -> np.ndarray:
        """Compute self-attention scores (Q·K^T).

        Args:
            data: Input of shape (seq_len, d_model).
            n: Sequence length.

        Returns:
            Attention weights of shape (seq_len, seq_len).
        """
        Q = data  # Simplified: Q = K = V = input
        scores = Q @ Q.T / np.sqrt(d_model)
        # Softmax (row-wise)
        exp_scores = np.exp(scores - scores.max(axis=1, keepdims=True))
        weights = exp_scores / exp_scores.sum(axis=1, keepdims=True)
        output = weights @ data
        return output

    seq_lengths = [64, 128, 256, 512, 1024, 2048, 4096]
    setup_attn = lambda n: rng.randn(n, d_model)

    analyzer_attn = ComplexityAnalyzer()
    analyzer_attn.measure(naive_attention, seq_lengths, setup_attn, n_runs=3)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].loglog(analyzer_attn.sizes, analyzer_attn.times, 'o-',
                    color=COLORS['blue'], linewidth=2, markersize=6,
                    label=f'Measured: O(n^{analyzer_attn.fitted_exponent:.2f})')

    # Reference lines
    n_ref = np.array(seq_lengths, dtype=float)
    scale = analyzer_attn.times[0] / (seq_lengths[0] ** 2)
    axes[0].loglog(n_ref, scale * n_ref ** 2, '--', color=COLORS['red'],
                    alpha=0.5, label='O(n²)')

    axes[0].set_xlabel('Sequence Length')
    axes[0].set_ylabel('Time (seconds)')
    axes[0].set_title(f'Self-Attention Complexity\nFitted: {analyzer_attn.classify()}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.2)

    # Memory: attention matrix is S × S
    memory_mb = [(s * s * 4) / (1024 ** 2) for s in seq_lengths]
    axes[1].semilogy(seq_lengths, memory_mb, 's-', color=COLORS['orange'],
                      linewidth=2, markersize=6)
    axes[1].axhline(1000, color=COLORS['red'], linestyle='--', alpha=0.5,
                     label='1 GB')
    axes[1].set_xlabel('Sequence Length')
    axes[1].set_ylabel('Attention Matrix Memory (MB)')
    axes[1].set_title('Attention Memory: O(S²)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.2)

    plt.suptitle('Transformer Attention: The Quadratic Bottleneck',
                  fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Attention matrix size (float32):')
    for s, m in zip(seq_lengths, memory_mb):
        print(f'  S={s:>5,}: {m:>10.2f} MB')


attention_complexity_demo()

### 3.3 Batch Size and ThroughputBatch size affects both training speed and memory. Larger batchesuse GPU parallelism better but require more memory.

In [ ]:
def batch_size_throughput() -> None:
    """Measure throughput (samples/sec) vs batch size."""
    rng = np.random.RandomState(SEED)
    d_in = 784
    d_hidden = 256
    d_out = 10

    W1 = rng.randn(d_in, d_hidden).astype(np.float32)
    W2 = rng.randn(d_hidden, d_out).astype(np.float32)

    def forward_pass(X: np.ndarray, n: int) -> np.ndarray:
        """Forward pass through 2-layer MLP.

        Args:
            X: Input batch.
            n: Batch size.

        Returns:
            Output predictions.
        """
        h = np.maximum(X @ W1, 0)  # ReLU
        return h @ W2

    batch_sizes = [1, 4, 16, 32, 64, 128, 256, 512, 1024]
    throughputs = []
    times_per_batch = []

    for bs in batch_sizes:
        X_batch = rng.randn(bs, d_in).astype(np.float32)
        mean_t, _ = time_function(forward_pass, X_batch, bs, n_runs=10)
        throughputs.append(bs / mean_t)
        times_per_batch.append(mean_t)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].semilogx(batch_sizes, [t / 1000 for t in throughputs], 'o-',
                      color=COLORS['blue'], linewidth=2, markersize=6)
    axes[0].set_xlabel('Batch Size')
    axes[0].set_ylabel('Throughput (K samples/sec)')
    axes[0].set_title('Throughput vs Batch Size')
    axes[0].grid(True, alpha=0.2)

    # Time per sample
    time_per_sample = [t / bs * 1e6 for t, bs in zip(times_per_batch, batch_sizes)]
    axes[1].semilogx(batch_sizes, time_per_sample, 's-',
                      color=COLORS['red'], linewidth=2, markersize=6)
    axes[1].set_xlabel('Batch Size')
    axes[1].set_ylabel('Time per Sample (μs)')
    axes[1].set_title('Per-Sample Cost vs Batch Size')
    axes[1].grid(True, alpha=0.2)

    plt.suptitle('Batch Size Effect on MLP Throughput (CPU)',
                  fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Observations:')
    print(f'  Batch=1:    {throughputs[0]:>10,.0f} samples/sec')
    print(f'  Batch=256:  {throughputs[6]:>10,.0f} samples/sec')
    print(f'  Speedup:    {throughputs[6] / throughputs[0]:.1f}x')
    print('  Larger batches amortize function call overhead and use BLAS more efficiently')


batch_size_throughput()

---## Part 4 — Evaluation & Analysis

### 4.1 Error Analysis: Common Complexity Mistakes

In [ ]:
def complexity_mistakes() -> None:
    """Demonstrate common complexity analysis mistakes."""
    rng = np.random.RandomState(SEED)

    print('=== Mistake 1: Ignoring constant factors ===')
    n = 10000
    a = rng.randn(n)

    # Both O(n), but very different constants
    t_simple, _ = time_function(lambda a, n: np.sum(a), a, n)
    t_complex, _ = time_function(
        lambda a, n: np.sum(np.exp(np.sin(np.log(np.abs(a) + 1)))), a, n
    )
    print(f'  Simple sum:   {t_simple*1e6:.1f} μs')
    print(f'  Complex sum:  {t_complex*1e6:.1f} μs')
    print(f'  Both O(n), but complex is {t_complex/t_simple:.1f}x slower!\n')

    print('=== Mistake 2: Amortized vs worst-case ===')
    # Python list append: O(1) amortized, O(n) worst case (reallocation)
    sizes_append = [1000, 10000, 100000, 1000000]
    for size in sizes_append:
        start = time.perf_counter()
        lst = []
        for i in range(size):
            lst.append(i)
        elapsed = time.perf_counter() - start
        print(f'  {size:>10,} appends: {elapsed*1000:.2f} ms '
              f'({elapsed/size*1e6:.2f} μs/op)')
    print('  Amortized O(1) per append — total is O(n)\n')

    print('=== Mistake 3: Hidden quadratic in string concatenation ===')
    sizes_str = [100, 500, 1000, 5000, 10000]
    times_concat = []
    times_join = []

    for size in sizes_str:
        words = ['word'] * size

        # O(n²) — string concatenation creates new string each time
        start = time.perf_counter()
        result = ''
        for w in words:
            result = result + w
        times_concat.append(time.perf_counter() - start)

        # O(n) — join pre-allocates
        start = time.perf_counter()
        result = ''.join(words)
        times_join.append(time.perf_counter() - start)
        _ = result

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.loglog(sizes_str, times_concat, 'o-', color=COLORS['red'],
               linewidth=2, label='str += str (O(n²))')
    ax.loglog(sizes_str, times_join, 's-', color=COLORS['green'],
               linewidth=2, label="''.join() (O(n))")
    ax.set_xlabel('Number of Strings')
    ax.set_ylabel('Time (seconds)')
    ax.set_title('Hidden Quadratic: String Concatenation')
    ax.legend()
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

    print('  At n=10,000:')
    print(f'    Concatenation: {times_concat[-1]*1000:.2f} ms')
    print(f'    Join:          {times_join[-1]*1000:.4f} ms')
    print(f'    Speedup:       {times_concat[-1]/times_join[-1]:.0f}x')


complexity_mistakes()

### 4.2 Quick Profiling TechniquesBefore optimizing, measure! Here are simple profiling approaches.

In [ ]:
def profiling_demo() -> None:
    """Demonstrate simple profiling techniques."""
    rng = np.random.RandomState(SEED)

    # Profile a mini ML pipeline
    n = 5000
    d = 50
    X = rng.randn(n, d)
    y = rng.randn(n)

    timings = {}

    # Step 1: Feature scaling
    start = time.perf_counter()
    X_mean = X.mean(axis=0)
    X_std = X.std(axis=0) + 1e-8
    X_norm = (X - X_mean) / X_std
    timings['Feature Scaling'] = time.perf_counter() - start

    # Step 2: Covariance matrix
    start = time.perf_counter()
    cov = (X_norm.T @ X_norm) / n
    timings['Covariance (X^TX)'] = time.perf_counter() - start

    # Step 3: Eigendecomposition
    start = time.perf_counter()
    eigvals, eigvecs = np.linalg.eigh(cov)
    timings['Eigendecomposition'] = time.perf_counter() - start

    # Step 4: PCA projection
    start = time.perf_counter()
    k = 10
    X_pca = X_norm @ eigvecs[:, -k:]
    timings['PCA Projection'] = time.perf_counter() - start

    # Step 5: Distance matrix
    start = time.perf_counter()
    dists = np.sum((X_pca[:, np.newaxis, :] - X_pca[np.newaxis, :, :]) ** 2, axis=2)
    timings['Pairwise Distances'] = time.perf_counter() - start
    _ = dists

    # Visualize as horizontal bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    steps = list(timings.keys())
    times_ms = [timings[s] * 1000 for s in steps]

    bars = axes[0].barh(steps, times_ms, color=COLOR_LIST[:len(steps)], alpha=0.7)
    axes[0].set_xlabel('Time (ms)')
    axes[0].set_title(f'Pipeline Profile (n={n}, d={d})')
    for bar, t in zip(bars, times_ms):
        axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
                      f'{t:.2f} ms', va='center', fontsize=9)
    axes[0].grid(True, alpha=0.2, axis='x')

    # Percentage breakdown
    total_time = sum(times_ms)
    percentages = [t / total_time * 100 for t in times_ms]
    axes[1].pie(percentages, labels=steps, autopct='%1.0f%%',
                 colors=COLOR_LIST[:len(steps)])
    axes[1].set_title(f'Time Breakdown\nTotal: {total_time:.1f} ms')

    plt.suptitle('Pipeline Profiling: Find the Bottleneck',
                  fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Profile summary:')
    for step, t_ms in zip(steps, times_ms):
        pct = t_ms / total_time * 100
        print(f'  {step:25s}: {t_ms:>8.2f} ms ({pct:>5.1f}%)')
    print(f'  {"TOTAL":25s}: {total_time:>8.2f} ms')

    bottleneck = steps[np.argmax(times_ms)]
    print(f'\nBottleneck: {bottleneck}')
    print('Optimize the bottleneck first — Amdahl\'s law!')


profiling_demo()

### 4.3 Algorithmic vs Hardware OptimizationA better algorithm always beats faster hardware. Let's demonstratewith a concrete example.

In [ ]:
def algorithm_vs_hardware() -> None:
    """Show that algorithmic improvement beats constant speedup."""
    # Problem: find if a sorted array contains a target
    rng = np.random.RandomState(SEED)

    sizes = [100, 1000, 10000, 100000, 1000000]
    times_linear = []
    times_binary = []

    for n in sizes:
        arr = np.sort(rng.randn(n))
        target = arr[n // 3]  # Known to exist

        # O(n) linear search
        def linear_search(arr: np.ndarray, n: int) -> int:
            """Linear search for target."""
            for i in range(len(arr)):
                if arr[i] == target:
                    return i
            return -1

        # O(log n) binary search
        def binary_search(arr: np.ndarray, n: int) -> int:
            """Binary search for target."""
            lo, hi = 0, len(arr) - 1
            while lo <= hi:
                mid = (lo + hi) // 2
                if arr[mid] == target:
                    return mid
                elif arr[mid] < target:
                    lo = mid + 1
                else:
                    hi = mid - 1
            return -1

        t_lin, _ = time_function(linear_search, arr, n)
        t_bin, _ = time_function(binary_search, arr, n)
        times_linear.append(t_lin)
        times_binary.append(t_bin)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.loglog(sizes, times_linear, 'o-', color=COLORS['red'],
               linewidth=2, label='Linear Search O(n)')
    ax.loglog(sizes, times_binary, 's-', color=COLORS['green'],
               linewidth=2, label='Binary Search O(log n)')

    # Show: even with 100x hardware speedup, O(n) still loses
    ax.loglog(sizes, [t / 100 for t in times_linear], '--', color=COLORS['orange'],
               linewidth=1.5, alpha=0.7, label='Linear + 100x faster hardware')

    ax.set_xlabel('Array Size n', fontsize=12)
    ax.set_ylabel('Time (seconds)', fontsize=12)
    ax.set_title('Algorithm > Hardware\n'
                  'Binary search on slow machine beats linear search on fast machine')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

    # Crossover point
    print(f'At n=1,000,000:')
    print(f'  Linear search:             {times_linear[-1]*1e6:>10.1f} μs')
    print(f'  Binary search:             {times_binary[-1]*1e6:>10.1f} μs')
    print(f'  Linear (100x HW speedup):  {times_linear[-1]*1e6/100:>10.1f} μs')
    print(f'  Binary still wins by:      {times_linear[-1] / 100 / times_binary[-1]:.1f}x')


algorithm_vs_hardware()

---## Part 5 — Summary & Lessons Learned### Key Takeaways- **Big-O notation** classifies algorithms by growth rate, ignoring constants.  Common ML complexities: linear regression $O(nd^2)$, k-NN prediction $O(nd)$,  self-attention $O(S^2 d)$, matrix inversion $O(d^3)$.- **Empirical measurement** is essential — time your code at different sizes  and fit the growth curve. The log-log slope directly gives the exponent.- **Vectorization** with NumPy/PyTorch gives 10–1000x speedups over Python  loops by leveraging BLAS, SIMD, and cache-friendly memory access. Always  vectorize inner loops.- **Memory matters** as much as time. Broadcasting creates large intermediate  tensors, attention matrices scale as $O(S^2)$, and Adam optimizer doubles  the parameter memory. Profile memory, not just speed.- **A better algorithm always wins** over faster hardware at scale.  Dropping from $O(n^2)$ to $O(n \log n)$ matters more than a 100x  hardware speedup for large $n$.### What's Next→ This completes **Module 01 — Mathematical & Programming Foundations**.You now have the tools for linear algebra, probability, information theory,calculus, optimization, and computational analysis. **Module 02 (SupervisedLearning)** applies these foundations to build real ML models.